# Exoplanet transit detection under stellar variability
**A reproducible TESS / Box Least Squares case study: WASP-18 and AU Mic**

**Question.** How reliably can Box Least Squares (BLS) identify a transit-like period when the time series contains additional stellar variability?

This notebook reorganises the original analysis in `exoplanet_1.ipynb`. It keeps both observed-system case studies and removes repeated cells. It **does not claim a new exoplanet detection**.

**Data and software:** public TESS photometry via Lightkurve; Python, NumPy and Matplotlib. Downloads need an internet connection.

**Reproduction:** install packages from `requirements.txt`, then use *Run All*. Results may differ with Lightkurve versions or archive products; the selected product metadata is printed below.

## 1. Load and inspect the WASP-18 data
For a transparent like-for-like experiment, we select the first product returned by the TESS search. This follows the original notebook's `download()` default; it is **not** a multi-sector analysis.

Flattening occurs **before** synthetic variability is added, so the following experiment tests how BLS reacts to a controlled injected signal rather than modelling realistic stellar astrophysics.

In [ ]:
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

def first_tess_product(target):
    results = lk.search_lightcurve(target, mission="TESS")
    if len(results) == 0:
        raise RuntimeError(f"No TESS light curves found for {target}")
    print(f"{target}: {len(results)} matching products; using the first result")
    print(results[:1])
    lightcurve = results[0].download()
    if lightcurve is None:
        raise RuntimeError(f"Could not download TESS light curve for {target}")
    print("Selected metadata:", {"sector": lightcurve.meta.get("SECTOR"),
           "author": lightcurve.meta.get("AUTHOR"),
           "target": lightcurve.meta.get("TARGETID")})
    return lightcurve

def prepare_lc(lc):
    # Same simple preprocessing strategy as the original notebook.
    return lc.remove_outliers().flatten()

def bls_period(lc):
    periodogram = lc.to_periodogram(method="bls")
    return periodogram, periodogram.period_at_max_power.to_value("day")

wasp_raw = first_tess_product("WASP-18")
wasp_clean = prepare_lc(wasp_raw)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
wasp_raw.normalize().plot(ax=axs[0], label="Original (normalised)")
wasp_clean.plot(ax=axs[1], label="Cleaned and flattened")
axs[0].set_title("WASP-18: TESS photometry")
axs[1].set_title("Preprocessed light curve")
plt.tight_layout()
plt.show()

## 2. Establish the transit-period baseline
The first BLS run provides a **within-notebook reference period**. It is not an independent measurement of the planet's ephemeris; this experiment compares changes relative to that baseline.

In [ ]:
pg_base, reference_period = bls_period(wasp_clean)
print(f"WASP-18 BLS baseline period: {reference_period:.8f} days")
fig, ax = plt.subplots(figsize=(9, 3.8))
pg_base.plot(ax=ax)
ax.set_title("WASP-18: baseline BLS periodogram")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9, 3.8))
wasp_clean.fold(period=pg_base.period_at_max_power).scatter(ax=ax, alpha=0.3, s=4)
ax.set_title("WASP-18: phase-folded at baseline period")
plt.tight_layout()
plt.show()

## 3. Inject controlled sinusoidal variability
Use the same two injected amplitudes as the original notebook: 0.003 and 0.010 in the flattened flux units, with a 2.5-day variability period. Compare both against the unmodified baseline.

This is a **stress test of the detection procedure**, not a physically calibrated stellar-variability model. A periodogram peak can reflect the injected variability rather than a transit.

In [ ]:
times = wasp_clean.time.value
base_flux = wasp_clean.flux.value
variability_period_days = 2.5
amplitudes = [0.0, 0.003, 0.010]

results = []
periodograms = {}
variants = {}
for amplitude in amplitudes:
    altered_flux = base_flux + amplitude * np.sin(
        2 * np.pi * times / variability_period_days
    )
    variant = lk.LightCurve(time=times, flux=altered_flux)
    periodogram, best_period = bls_period(variant)
    variants[amplitude] = variant
    periodograms[amplitude] = periodogram
    results.append({
        "injected_amplitude": amplitude,
        "best_BLS_period_days": best_period,
        "difference_from_baseline_pct": 100 * abs(best_period - reference_period) / reference_period,
        "baseline_recovered_within_2pct": abs(best_period - reference_period) / reference_period < 0.02
    })

results_df = pd.DataFrame(results)
display(results_df.round(5))

In [ ]:
fig, axes = plt.subplots(len(amplitudes), 2, figsize=(12, 9))
for i, amplitude in enumerate(amplitudes):
    variant = variants[amplitude]
    pg = periodograms[amplitude]
    variant.plot(ax=axes[i, 0], label=f"Injected amplitude = {amplitude:.3f}")
    pg.plot(ax=axes[i, 1])
    axes[i, 0].set_title(f"Flux: amplitude {amplitude:.3f}")
    axes[i, 1].set_title(f"BLS: amplitude {amplitude:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Markdown, display
first_failed = results_df.loc[~results_df["baseline_recovered_within_2pct"]]
if first_failed.empty:
    display(Markdown(
        "With these tested amplitudes and a 2% tolerance, all BLS peaks remain near "
        "the baseline period. This does not establish robustness outside this specific test."
    ))
else:
    row = first_failed.iloc[0]
    display(Markdown(
        f"**Observation:** At injected amplitude **{row['injected_amplitude']:.3f}**, "
        f"the strongest BLS peak shifts to **{row['best_BLS_period_days']:.4f} days**, "
        f"rather than the baseline **{reference_period:.4f} days**. "
        "The signal in this experiment has become harder to identify correctly."
    ))

## 4. AU Mic: an active-star case study
Repeat the simple BLS workflow for the first available TESS product of AU Mic. **Do not treat the strongest BLS peak as proof of an orbital period**: stellar variability, instrumental features and preprocessing can all influence periodogram maxima.

A rigorous detection study would use several sectors, stellar-rotation and flare treatment, transit-duration checks and independent ephemerides.

In [ ]:
au_raw = first_tess_product("AU Mic")
au_clean = prepare_lc(au_raw)
au_pg, au_peak = bls_period(au_clean)
print(f"AU Mic: strongest BLS periodogram peak = {au_peak:.6f} days")
fig, axs = plt.subplots(3, 1, figsize=(10, 9))
au_raw.normalize().plot(ax=axs[0], label="Raw (normalised)")
au_clean.plot(ax=axs[1], label="Flattened")
au_pg.plot(ax=axs[2])
axs[0].set_title("AU Mic: raw photometry")
axs[1].set_title("AU Mic: flattened light curve")
axs[2].set_title("AU Mic: BLS periodogram (peak is not a confirmed transit)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
au_clean.fold(period=au_pg.period_at_max_power).scatter(ax=ax, alpha=0.3, s=4)
ax.set_title(f"AU Mic folded at BLS peak ({au_peak:.4f} d); not a validated transit period")
plt.tight_layout()
plt.show()

## 5. Interpretation and limitations
1. **WASP-18 baseline:** BLS can identify an approximately 0.94-day period in the chosen preprocessed light curve.
2. **Controlled perturbation:** Injecting a sufficiently strong 2.5-day sinusoidal component can shift the dominant BLS peak. The notebook quantifies where that happens in this specific dataset.
3. **AU Mic:** A single-sector, simply flattened light curve produces a BLS maximum; it should not be reported as the planetary period without further validation.
4. **Scope:** The test uses a single downloaded product per target, no injection-recovery statistics over many noise realisations, and no rigorous detrending benchmark. It does not measure completeness, false-alarm rates, or transit detectability across all stellar types.
5. **Future work:** multi-sector light curves, rotation-aware detrending, flare masks, transit-model fits and Monte Carlo injection-recovery tests.

**Original notebook:** `exoplanet_1.ipynb` remains in the repository with its original saved plots and execution outputs. This clean version is the preferred reproducible starting point.